# BIRD-SQL Evaluation — EX & VES cho các model Ollama

Chạy `evaluation.py` (Execution Accuracy) và `evaluation_ves.py` (Valid Efficiency Score)
cho 3 model đã có kết quả: `deepseek-coder-v2`, `gemma3`, `qwen2.5-coder`.

Gọi qua `!python` (subprocess thật) thay vì `import` trực tiếp — 2 script này dùng
`multiprocessing.Pool`, import thẳng vào kernel Jupyter trên Windows sẽ lỗi spawn.

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
LLM_DIR = PROJECT_ROOT / "src" / "DAMO-ConvAI" / "bird" / "llm"
BIRD_DATA_DIR = PROJECT_ROOT / "src" / "DAMO-ConvAI" / "bird" / "data"

# model hiển thị -> (thư mục w/ knowledge, thư mục w/o knowledge hoặc None nếu chưa có)
MODEL_FOLDERS = {
    "DeepSeek-Coder-V2": ("ollama_deepseekcoder_output", "ollama_deepseekcoder_0_output"),
    "Gemma3":            ("ollama_gemma3_output",        "ollama_gemma3_0_output"),
    "Qwen2.5-Coder":     ("ollama_gwen25_output",         "ollama_gwen25_0_output"),
    "GPT-3.5-Turbo":     ("openai_output",                "openai_0_output"),
}

for name, (kg_dir, nokg_dir) in MODEL_FOLDERS.items():
    for label, d in [("w/ knowledge", kg_dir), ("w/o knowledge", nokg_dir)]:
        if d is None:
            print(f"{name:20} {label:15} -> CHƯA CHẠY (bỏ qua)")
            continue
        p = LLM_DIR / "exp_result" / d / "predict_dev.json"
        print(f"{name:20} {label:15} -> {'OK' if p.exists() else 'MISSING'}  {p}")

GPT-3.5-Turbo        w/ knowledge    -> OK  d:\AI Learning\Paper\LLM\src\DAMO-ConvAI\bird\llm\exp_result\openai_output\predict_dev.json
GPT-3.5-Turbo        w/o knowledge   -> OK  d:\AI Learning\Paper\LLM\src\DAMO-ConvAI\bird\llm\exp_result\openai_0_output\predict_dev.json


In [2]:
def run_eval_script(script_name: str, predicted_dir_name: str, num_cpus: int = 4, meta_time_out: float = 30.0,
                     extra_args=None) -> str:
    """Run evaluation.py or evaluation_ves.py as a subprocess (avoids Windows multiprocessing-in-kernel issues)."""
    cmd = [
        sys.executable, f"src/{script_name}",
        "--db_root_path", str(BIRD_DATA_DIR / "dev_databases") + os.sep,
        "--predicted_sql_path", str(LLM_DIR / "exp_result" / predicted_dir_name) + os.sep,
        "--ground_truth_path", str(LLM_DIR / "data") + os.sep,
        "--data_mode", "dev",
        "--num_cpus", str(num_cpus),
        "--meta_time_out", str(meta_time_out),
        "--mode_gt", "gt",
        "--mode_predict", "gpt",
        "--diff_json_path", str(BIRD_DATA_DIR / "dev.json"),
    ]
    if extra_args:
        cmd += extra_args
    result = subprocess.run(cmd, cwd=str(LLM_DIR), capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[{predicted_dir_name} / {script_name}] STDERR:\n", result.stderr[-2000:])
    return result.stdout


SCORE_RE = re.compile(r"(accuracy|ves)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)")


def parse_score(stdout: str):
    match = SCORE_RE.search(stdout)
    if not match:
        return None
    _, simple, moderate, challenging, total = match.groups()
    return [float(simple), float(moderate), float(challenging), float(total)]

In [3]:
# EX Score
ex_scores = {}

for name, (kg_dir, nokg_dir) in MODEL_FOLDERS.items():
    ex_scores[name] = {}
    for label, d in [("w_kg", kg_dir), ("nokg", nokg_dir)]:
        if d is None:
            ex_scores[name][label] = None
            continue

        print(f"\n===== {name} ({label}) : EX =====")
        ex_out = run_eval_script("evaluation.py", d)
        print(ex_out[-500:])
        ex_scores[name][label] = parse_score(ex_out)


===== GPT-3.5-Turbo (w_kg) : EX =====
  simple               moderate             challenging          total               
count                925                  464                  145                  1534                
======================================    ACCURACY    =====================================
accuracy             57.51                35.56                24.83                47.78               
Finished evaluation


===== GPT-3.5-Turbo (w_kg) : VES (iterate_num=5) =====
  simple               moderate             challenging          total               
count                925                  464                  145                  1534                
=========================================    VES   ========================================
ves                  61.23                48.30                24.87                53.88               
Finished evaluation


===== GPT-3.5-Turbo (nokg) : EX =====
  simple               moderate             chal

In [4]:
# Print EX
def get_total(score):
    return score[3] if score is not None else None

ex_rows = {}
for name in MODEL_FOLDERS:
    ex_rows[name] = {
        "EX_wo_knowledge": get_total(ex_scores[name]["nokg"]),
        "EX_w_knowledge": get_total(ex_scores[name]["w_kg"]),
    }

ex_summary = pd.DataFrame(ex_rows).T
ex_summary

,EX_wo_knowledge,EX_w_knowledge,VES_wo_knowledge,VES_w_knowledge
GPT-3.5-Turbo,31.23,47.78,33.88,53.88


In [ ]:
OUT_EX_CSV = LLM_DIR / "exp_result" / "summary_ex.csv"
ex_summary.to_csv(OUT_EX_CSV)
print("Saved:", OUT_EX_CSV)

In [ ]:
# VES Score
ves_scores = {}

for name, (kg_dir, nokg_dir) in MODEL_FOLDERS.items():
    ves_scores[name] = {}
    for label, d in [("w_kg", kg_dir), ("nokg", nokg_dir)]:
        if d is None:
            ves_scores[name][label] = None
            continue

        print(f"\n===== {name} ({label}) : VES (iterate_num=5) =====")
        ves_out = run_eval_script("evaluation_ves.py", d, extra_args=["--iterate_num", "5"])
        print(ves_out[-500:])
        ves_scores[name][label] = parse_score(ves_out)

In [ ]:
# Print VES
ves_rows = {}
for name in MODEL_FOLDERS:
    ves_rows[name] = {
        "VES_wo_knowledge": get_total(ves_scores[name]["nokg"]),
        "VES_w_knowledge": get_total(ves_scores[name]["w_kg"]),
    }

ves_summary = pd.DataFrame(ves_rows).T
ves_summary

In [ ]:
OUT_VES_CSV = LLM_DIR / "exp_result" / "summary_ves.csv"
ves_summary.to_csv(OUT_VES_CSV)
print("Saved:", OUT_VES_CSV)